<a href="https://colab.research.google.com/github/atletisimo/THREAT_DETECTION_LOGS/blob/main/finki_cyber_threat_logs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("aryan208/cybersecurity-threat-detection-logs")

ИМПОРТИРАЊЕ НА ПОТРЕБНИТЕ БИБЛИОТЕКИ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.neighbors import kneighbors_graph
import warnings

warnings.filterwarnings('ignore')

DATA LOADING AND PREPROCESSING

In [ ]:
print("="*80)
print("LOADING AND PREPROCESSING DATA")
print("="*80)  #ова е само за визуелен изглед

df=pd.read_csv('/root/.cache/kagglehub/datasets/aryan208/cybersecurity-threat-detection-logs/versions/1/cybersecurity_threat_detection_logs.csv')

print("Број на редици:", df.shape[0])
print("\n ")
print("Број на колони:",df.shape[1])
print("\n ")
print("Први 5 редици:",df.head())
print("\n ")
print("Податоци за dataset-от:")
print("\n ")
print(df.info())
print("\n ")
#Ги оделуваме features и label
# Identify non-numeric columns to drop before scaling
non_numeric_cols = df.select_dtypes(include=['object']).columns.tolist()
# Ensure 'threat_label' is not included if it's already separated into 'y'
if 'threat_label' in non_numeric_cols:
    non_numeric_cols.remove('threat_label')

X=df.drop(columns=non_numeric_cols + ["threat_label"],axis=1) # Drop non-numeric and the label column
print('Features (X) after dropping non-numeric columns:')
print(X.head())
print("\n ")
y=df['threat_label'] #само label-от
print('Labels (y):')
print(y.head())
#Поделба на множеството за тренирање и тестирање => 20% за тест ,80% за тренинг
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

print(f"\nГолемина на тренинг множество: {X_train.shape[0]}")
print(f"Големина на тест множество: {X_test.shape[0]}")

ПОМОШНИ ФУНКЦИИ


In [ ]:
def calculate_metrics(y_true, y_pred):
    """Пресметка на: ACC, FAR, and UND"""
    cm = confusion_matrix(y_true, y_pred)

    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]

    ACC = ((TP + TN) / (TP + TN + FP + FN)) * 100

    FAR = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0

    UND = (FN / (FN + TP)) * 100 if (FN + TP) > 0 else 0

    return {
        'TN': TN, 'FP': FP, 'FN': FN, 'TP': TP,
        'Accuracy': ACC, 'FAR': FAR, 'UND': UND
    }


def print_metrics(model_name, metrics):
    print(f"\n{model_name} Резултати:")
    print("-" * 60)
    print(f"Confusion Matrix:")
    print(f"  TN: {metrics['TN']:6d}  |  FP: {metrics['FP']:6d}")
    print(f"  FN: {metrics['FN']:6d}  |  TP: {metrics['TP']:6d}")
    print(f"\nМетрики за перформанс:")
    print(f"  Accuracy (ACC):        {metrics['Accuracy']:.2f}%")
    print(f"  False Alarm Rate (FAR): {metrics['FAR']:.2f}%")
    print(f"  Un-Detection Rate (UND): {metrics['UND']:.2f}%")


results = {}

LOGISTIC REGRESSION



In [ ]:
print("\n" + "=" * 70)
print("Logistic regression")
print("=" * 70)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

lr_metrics = calculate_metrics(y_test, lr_pred)
results['Logistic Regression'] = lr_metrics
print_metrics('Logistic Regression', lr_metrics)

RANDOM FOREST


In [ ]:
print("\n" + "=" * 70)
print("RANDOM FOREST")
print("=" * 70)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_metrics = calculate_metrics(y_test, rf_pred)
results['Random Forest'] = rf_metrics
print_metrics('Random Forest', rf_metrics)


RANDOM FOREST


NAIVE BAYES


In [ ]:
print("\n" + "=" * 70)
print("NAIVE BAYES")
print("=" * 70)

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_pred = nb_model.predict(X_test_scaled)

nb_metrics = calculate_metrics(y_test, nb_pred)
results['Naive Bayes'] = nb_metrics
print_metrics('Naive Bayes', nb_metrics)

KNN

In [ ]:
print("\n" + "=" * 70)
print("K-NEAREST NEIGHBORS (KNN)")
print("=" * 70)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

knn_metrics = calculate_metrics(y_test, knn_pred)
results['KNN'] = knn_metrics
print_metrics('KNN', knn_metrics)

DECISION TREE

In [ ]:
print("\n" + "=" * 70)
print("DECISION TREE")
print("=" * 70)

dt_model = DecisionTreeClassifier(random_state=42, max_depth=20)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

dt_metrics = calculate_metrics(y_test, dt_pred)
results['Decision Tree'] = dt_metrics
print_metrics('Decision Tree', dt_metrics)

СПОРЕДБА НА РЕЗУЛТАТИ

In [ ]:
print("\n" + "=" * 70)
print("GENERATING COMPARISON CHARTS")
print("=" * 70)

# Prepare data for visualization
models = list(results.keys())
accuracy = [results[m]['Accuracy'] for m in models]
far = [results[m]['FAR'] for m in models]
und = [results[m]['UND'] for m in models]

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': models,
    'Accuracy (%)': accuracy,
    'FAR (%)': far,
    'UND (%)': und
})

print("\nSummary Table:")
print(comparison_df.to_string(index=False))

plt.figure(figsize=(12, 6))
bars = plt.bar(models, accuracy, color=['#2E86AB', '#E63946', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#457B9D'])
plt.ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Accuracy Comparison Across ML Algorithms', fontsize=14, fontweight='bold')
plt.ylim([0, 105])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height,
             f'{height:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
bars = plt.bar(models, far, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E'])
plt.ylabel('False Alarm Rate (%)', fontsize=12, fontweight='bold')
plt.title('False Alarm Rate (FAR) Comparison Across ML Algorithms', fontsize=14, fontweight='bold')
plt.ylim([0, max(far) * 1.2])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height,
             f'{height:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
bars = plt.bar(models, und, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E'])
plt.ylabel('Un-Detection Rate (%)', fontsize=12, fontweight='bold')
plt.title('Un-Detection Rate (UND) Comparison Across ML Algorithms', fontsize=14, fontweight='bold')
plt.ylim([0, max(und) * 1.2])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height,
             f'{height:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
axes[0].bar(models, accuracy, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E'])
axes[0].set_ylabel('Accuracy (%)', fontweight='bold')
axes[0].set_title('Accuracy', fontweight='bold', fontsize=12)
axes[0].set_ylim([0, 105])
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# FAR
axes[1].bar(models, far, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E'])
axes[1].set_ylabel('False Alarm Rate (%)', fontweight='bold')
axes[1].set_title('False Alarm Rate (FAR)', fontweight='bold', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# UND
axes[2].bar(models, und, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E'])
axes[2].set_ylabel('Un-Detection Rate (%)', fontweight='bold')
axes[2].set_title('Un-Detection Rate (UND)', fontweight='bold', fontsize=12)
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Complete Performance Comparison Across ML Algorithms',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
heatmap_data = comparison_df.set_index('Model')[['Accuracy (%)', 'FAR (%)', 'UND (%)']]
sns.heatmap(heatmap_data.T, annot=True, fmt='.2f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Percentage (%)'}, linewidths=1)
plt.title('Performance Metrics Heatmap', fontsize=14, fontweight='bold')
plt.ylabel('Metrics', fontweight='bold')
plt.xlabel('ML Algorithms', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE!")
print("=" * 70)